# 03 - Geographic analysis (ENEM 2025)

**Questions answered here**
- Which states have the highest and lowest mean scores?
- Which region performs best overall and in each knowledge area?
- How different are states within the same region?
- Which states are the most balanced?

**Method notes**
- Source: INEP `RESULTADOS_2025.csv` (see notebook 02 for how the data was inspected and cleaned).
- Each area is averaged over participants marked present in that area. Blank objective answer sheets (score 0 with all answers blank) are excluded.
- Geography is the state where the exam was taken (`SG_UF_PROVA`), not the state of residence.
- Scales differ by area, so areas are never compared by raw points: comparisons are made within an area.

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src import stats, plots
from src.config import AREAS, RAW_DIR, UF_REGION
from src.load import prepare_dataset

df, school = prepare_dataset()
print(df.shape)

## 2. National means
Means over participants present in each area, excluding blank answer sheets.

In [ ]:
national = stats.national_mean_std(df)
national_table = pd.DataFrame(national, index=["mean", "std"]).round(1)
print(national_table.to_string())

counts = {a: int((df[f"TP_PRESENCA_{a}"] == 1).sum()) for a in AREAS}
print(counts)

## 3. Means by region

In [ ]:
region_means = stats.mean_table(df, "REGION")
print(region_means.round(1).sort_values("LC", ascending=False).to_string())

## 4. Means by state

In [ ]:
state_means = stats.mean_table(df, "SG_UF_PROVA")
print(state_means.round(1).sort_values("LC", ascending=False).to_string())

### Highest and lowest state in each column

In [ ]:
columns = AREAS + ["ESSAY_WITH_ZEROS", "ESSAY_VALID_ONLY"]
rows = []
for column in columns:
    ordered = state_means[column].sort_values(ascending=False)
    rows.append({
        "column": column,
        "highest": f"{ordered.index[0]} {ordered.iloc[0]:.1f}",
        "second": f"{ordered.index[1]} {ordered.iloc[1]:.1f}",
        "lowest": f"{ordered.index[-1]} {ordered.iloc[-1]:.1f}",
    })
print(pd.DataFrame(rows).to_string(index=False))

## 5. Absence rates by state
State means describe only the participants who showed up. If absence differs a lot between states, the means describe different shares of each state's registered population.

In [ ]:
absence = stats.absence_rates(df, "SG_UF_PROVA")
summary = state_means[["LC", "CN"]].join(absence).round(1)
print(summary.sort_values("absent_LC_%", ascending=False).to_string())
print()
print(summary.corr().round(2))

## 6. Spread inside each state (LC)
The standard deviation is the spread of individual scores. It is the yardstick for how large the differences between states are.

In [ ]:
spread = stats.score_spread(df, "SG_UF_PROVA", "LC").round(1)
print(spread.sort_values("mean", ascending=False).to_string())
print()
top, bottom = spread["mean"].max(), spread["mean"].min()
print("Highest minus lowest state mean:", round(top - bottom, 1))
print("National std of individual scores:", round(national["LC"][1], 1))

## 7. Differences between states of the same region (LC)

In [ ]:
print(stats.range_within_group(state_means, "LC", UF_REGION).round(1).to_string())

## 8. Profile across areas (z-scores)
Each state mean is expressed in standard deviations of individual scores above or below the national mean **of the same area**. This describes relative position within each area; it does not claim that areas measure the same thing.

In [ ]:
z = stats.state_zscores(state_means, national)
print(z.round(2).sort_values("range").to_string())

In [ ]:
fig = plots.zscore_heatmap(z)
plots.save_figure(fig, "heatmap_state_area_z.png")
plt.show()

## 9. Mean LC score by state

In [ ]:
national_lc = national["LC"][0]
fig = plots.state_dot_plot(state_means, area="LC", national_mean=national_lc)
plots.save_figure(fig, "lc_by_state.png")
plt.show()

## 10. Map (requires the IBGE state shapefile)

In [ ]:
import geopandas as gpd

shapefiles = list(RAW_DIR.parent.glob("ibge_uf_2025/*.shp"))
if shapefiles:
    shapes = gpd.read_file(shapefiles[0])
    fig = plots.state_map(
        shapes, state_means, "LC",
        "Mean LC score by state of exam (participants present)",
    )
    plots.save_figure(fig, "map_lc_by_state.png")
    plt.show()
else:
    print("Shapefile not found in data/raw/ibge_uf_2025")

## 11. Findings

**Observed**
- The Southeast has the highest mean in all four objective tests and the North has the lowest. The South is within about 1 to 3 points of the Southeast in the objective tests. In the essay, the Center-West is slightly above the South when zeros are included.
- Which state is "highest" depends on the column: the answer changes between CN/CH, LC/MT and the two essay averages (see section 4).
- Within-region differences in LC are smallest in the South and largest in the Northeast and Center-West (section 7). The Federal District stands well above its Center-West neighbors.
- A state is usually above (or below) the national mean in all four areas. Differences between areas inside one state are small, at most about 0.2 standard deviations (section 8).
- The gap between the highest and lowest state mean in LC is about the size of the standard deviation of individual scores. There is far more variation among people inside a state than between states.
- Absence rates differ a lot between states (section 5).

**Hypotheses (not tested here)**
- Differences in who registers and who shows up may explain part of the state differences.
- Some of the essay differences between states may come from different shares of invalid (zero) essays.

**What these results do not show**
- They do not show that education quality differs between states: the data compare exam takers, not residents, and there is no socioeconomic control.
- Differences of 1 to 2 points between neighboring states are not meaningful given the spread of individual scores.
- Raw scores of different areas must not be compared with each other.